# 07 - Risk levels and dashboard data

**Goal:** define the High / Medium / Low risk levels and build the data file read by the dashboard.

**Rules (fixed before looking at the test results)**
- High: probability >= 0.66 (the final decision threshold).
- Medium: probability >= `t_medium`, the largest threshold at which at least 90% of the bug-inducing commits of the validation year are still at or above it. Computed with the model trained on 2003-2015, like the 0.66 threshold.
- Low: everything below.
- `priority` = probability / (lines changed + 1): the review order that worked best in the effort-aware evaluation.

In [1]:
import sys
sys.path.append("..")

import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import recall_score

from src.data.split import load_data, time_split, FEATURES, LABEL
from src.models.builders import make_lightgbm

df = load_data()
train, validation, test, late = time_split(df)

with open("../results/04_lightgbm_params.json") as f:
    params = json.load(f)

model_2015 = make_lightgbm(**params).fit(train[FEATURES], train[LABEL])
p_val = model_2015.predict_proba(validation[FEATURES])[:, 1]

grid = np.arange(0.05, 0.96, 0.01)
recalls = pd.Series({round(t, 2): recall_score(validation[LABEL], p_val >= t) for t in grid})
t_medium = recalls[recalls >= 0.90].index.max()
t_medium

np.float64(0.32)

## 1. Risk levels on the validation year

**Goal:** check what the three levels mean: for each level, how many commits fall in it, how many of them are bug-inducing, and what share of all bug-inducing commits it holds. Low should hold at most about 10% of the bug-inducing commits by construction.

In [2]:
T_HIGH = 0.66

def level(p):
    if p >= T_HIGH:
        return "High"
    if p >= t_medium:
        return "Medium"
    return "Low"

def level_table(part, proba):
    levels = pd.Series(proba, index=part.index).apply(level)
    grouped = part.groupby(levels)[LABEL]
    table = pd.DataFrame({
        "commits": grouped.size(),
        "bug share in level": grouped.mean(),
        "share of all commits": grouped.size() / len(part),
        "share of all bug-inducing commits": grouped.sum() / part[LABEL].sum(),
    })
    return table.reindex(["High", "Medium", "Low"])

level_table(validation, p_val).round(3)

,commits,bug share in level,share of all commits,share of all bug-inducing commits
High,3408,0.578,0.321,0.638
Medium,3343,0.248,0.315,0.268
Low,3855,0.075,0.363,0.094


**Observation**
- `t_medium` is 0.32: at this threshold 90% of the bug-inducing commits of the validation year are still at or above it.
- Validation year (model trained on 2003-2015): High holds 32.1% of the commits with a bug share of 57.8% and 63.8% of all bug-inducing commits; Medium holds 31.5% of the commits (bug share 24.8%, 26.8% of the bugs); Low holds 36.3% of the commits (bug share 7.5%) and 9.4% of the bugs, below the 10% target.

## 2. Dashboard data

**Goal:** take the final model's explanations for the test year (`results/06_test_explanations.csv`), add the risk level, the review priority and a readable date, merge the three Hadoop repository names, check the levels on the test year (descriptive only) and save the file that the dashboard reads: `dashboard/data/commits.csv`.

In [4]:
data = pd.read_csv("../results/06_test_explanations.csv")
data["project"] = data["project"].replace({
    "apache/hadoop": "hadoop", "apache/hadoop-hdfs": "hadoop", "apache/hadoop-mapreduce": "hadoop"})
data["risk_level"] = data["probability"].apply(level)
data["priority"] = data["probability"] / (data["value_la"] + data["value_ld"] + 1)
data["date"] = pd.to_datetime(data["author_date"], unit="s")

Path("../dashboard/data").mkdir(parents=True, exist_ok=True)
data.round(5).to_csv("../dashboard/data/commits.csv", index=False)

print(data.shape)
level_table(test, data["probability"].to_numpy()).round(3)

(10187, 33)


C:\Users\pckala.org\AppData\Local\Temp\ipykernel_82884\3050871228.py:9: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  data.round(5).to_csv("../dashboard/data/commits.csv", index=False)


,commits,bug share in level,share of all commits,share of all bug-inducing commits
High,3068,0.578,0.301,0.639
Medium,3328,0.226,0.327,0.271
Low,3791,0.066,0.372,0.090


**Observation (descriptive; nothing was changed after seeing it)**
- Test year: High holds 30.1% of the commits with a bug share of 57.8% and 63.9% of all bug-inducing commits; Medium holds 32.7% (bug share 22.6%, 27.1% of the bugs); Low holds 37.2% (bug share 6.6%) and 9.0% of the bugs. The pattern is the same as on the validation year, so the levels carry over to unseen commits.
- Compared with the overall bug share of the test year (27.2%), the bug share is about 2.1 times higher in High and about 0.24 times in Low.
- About 265 commits of the Low level add no lines; they cannot be labeled bug-inducing by construction, so they push the Low bug share down slightly (to about 7% if they are left out; approximate).
- The bug share of Medium (22.6%) is below the overall share, so Medium means "in between / uncertain", not "elevated risk".

**Decision:** use these levels in the dashboard. Describe Medium as "uncertain", and describe High and Low by what they hold: High finds about 64% of the bug-inducing commits among 30% of the commits.

## Summary

- Levels: High (score >= 0.66), Medium (0.32 to 0.66), Low (< 0.32). The Medium limit keeps at least 90% of the validation bug-inducing commits at or above it.
- `dashboard/data/commits.csv` (10,187 test commits) holds the score, level, review priority and SHAP values for the dashboard.
- Next: the Django dashboard reads this file.